In [1]:

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from nltk.corpus import stopwords
from bertopic.representation import MaximalMarginalRelevance, PartOfSpeech, LangChain, KeyBERTInspired
from bertopic.vectorizers import  ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from collections import Counter
import plotly.io as pio
import re
pio.renderers.default = "vscode"
stoplist = list(set(stopwords.words('english')))

KeyboardInterrupt: 

In [ ]:
import os
import sys

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

In [ ]:
with open(project_root + "/params.json", 'r') as f:
    PARAMS = json.load(f)

In [ ]:
from data_gathering.utils.util import clean_text
from classification.utils_finetune import load_dataset, split_dataset

In [ ]:
model_id = PARAMS["classi_finetune_model"]
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

df = pd.read_csv("data/themes_from_thesis.csv", sep="\t")
primary_texts = [clean_text(x) for x in df["EXCERPT"]]

In [ ]:
thesis_df = pd.read_csv("xai/data/themes_from_thesis.csv", sep="\t")
themes = thesis_df["Themes"]
all_themes = []
for x in themes:
    all_themes.extend(str(x).split("# "))
Counter(all_themes)

In [ ]:
seed_words = []
for x in themes:
    try:
        words = [re.sub(r"\s+", "-", i.lstrip().rstrip().replace("&", "").replace(":", "").replace("(", "").replace(")", "").replace("/","-")) for i in x.split("#") if i!=""]
        if len(words)>1:
            seed_words.append([re.sub("-+", "-",i) for i in words])
    except:
        continue

In [ ]:
seed_words = list(set(map(lambda i: tuple(sorted(i)), seed_words)))

In [ ]:
seed_words = [list(x) for x in seed_words]

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(PARAMS["sentence_model"], model_kwargs={"torch_dtype": "float16"})

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()

    tokens = [t for t in tokens if t not in stoplist]
    return " ".join(tokens)

val_data["clean_text"] = val_data["text"].apply(clean_text)
thesis_df["clean_text"] = thesis_df["EXCERPT"].apply(clean_text)

In [ ]:
texts = thesis_df["clean_text"]

In [ ]:
embeddings = embedding_model.encode(list(texts), show_progress_bar=True)

In [ ]:
params = {  
            # TFIDF
            "reduce_frequent_words": True, "bm25_weighting": False,
            "seed_words": [],
            "seed_multiplier": 4,
            # UMAP
            "n_neighbors": 10, "n_components": 5, "min_dist": 0.0, "metric_umap": "cosine", "random_state": 42,
            # HDBSCAN (change min_cluster_size for more/less topics?, default is 10, recommended to only increase above 10)
            "min_cluster_size":2, "metric_hbd": "euclidean", "cluster_selection_method": "eom", "prediction_data": True,
            # Vectorizer model
            "stop_words": "english", "min_df": 1, "ngram_range": (1,4),
            # Representation models
            "diversity": 0.4
         }

In [ ]:
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=params["reduce_frequent_words"], bm25_weighting=params["bm25_weighting"],
                                     seed_words=params['seed_words'], seed_multiplier=params["seed_multiplier"])



In [ ]:

umap_model = UMAP(n_neighbors=params["n_neighbors"], 
                  n_components=params["n_components"], 
                  min_dist=params["min_dist"], 
                  metric=params["metric_umap"], 
                  random_state=params["random_state"])



In [ ]:
hdbscan_model = HDBSCAN(min_cluster_size=params["min_cluster_size"],
                        metric=params["metric_hbd"], 
                        cluster_selection_method=params["cluster_selection_method"], 
                        prediction_data=params["prediction_data"])

In [ ]:
vectorizer_model = CountVectorizer(stop_words=params["stop_words"], 
                                   min_df=params["min_df"], 
                                   ngram_range=params["ngram_range"])



In [ ]:

representation_models = [
                            MaximalMarginalRelevance(diversity=params["diversity"]),
                             KeyBERTInspired(),
                            # PartOfSpeech("en_core_web_sm"),
                            ]

In [ ]:
topic_model = BERTopic(

    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    # representation_model=representation_models,
    top_n_words=10,
    verbose=True,
    ctfidf_model=ctfidf_model,
    # nr_topics="auto",
    calculate_probabilities=True,
    seed_topic_list=seed_words,
)

# Train model
topics, probs = topic_model.fit_transform(texts, embeddings)
# new_topics = topic_model.reduce_outliers(texts, topics, probabilities=probs, threshold=0.03, strategy="probabilities")
# topic_model.update_topics(texts, topics=new_topics)


In [ ]:
# topics_per_class = topic_model.topics_per_class(texts, classes=labels)

In [ ]:
topic_info = topic_model.get_topic_info()
for idx, row in topic_info.iterrows():
    print()
    print(row["Name"])
    print()
    print(row["Representation"])
    print()
    print(row["Representative_Docs"])
    print()
    print("*"*30)

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.visualize_topics()

In [ ]:
# df["topics"] = [topic_model.topic_labels_[x] for x in topics]
# df["label"] = [id2label[i] for i in list(df["label"])]

In [ ]:
# len(df["topics"].unique())

In [ ]:
# df.head()

In [ ]:
# normalized_counts = pd.crosstab(
#     df['label'],
#     df['topics'],
#     normalize='index'   # normalize per label (row-wise)
# )
#
# print(normalized_counts)

In [ ]:
# import matplotlib.pyplot as plt
#
# normalized_counts.plot(
#     kind='barh',
#     stacked=True,
#     figsize=(10, 6),
#     colormap='tab20'
# )
#
# plt.xlabel("Proportion")
# plt.title("Topic Distribution per Label")
# plt.tight_layout()
# plt.show()